# Day 08 — structlog: Structured Logging

**Phase 1 · Module 1: LangSmith / Observability** · ~30 min

### What I practice today
- [ ] See why `print()` and plain string logs fall apart the moment you need to *search* them
- [ ] Emit **structured events** with `structlog` — a message plus **key=value** fields
- [ ] Switch to a **JSON renderer** so logs are machine-readable (CloudWatch, Datadog, etc.)
- [ ] **Bind** context (`trace_id`, `session_id`) once so it rides along on *every* later log line
- [ ] Use **context vars** so request-scoped fields reach deep helper functions with no plumbing
- [ ] Log an **exception** with full structured context
- [ ] Bridge: this is the same "one event = structured record" idea behind LangSmith traces (today's LangGraph notebook)


## 1. The everyday problem — a log line you can't search

Picture a bank's payment service. Someone taps "Send £40 to Mum". You want a log of it. The obvious first move is `print()` with an f-string:

In [1]:
# The "just glue it into a sentence" approach.
def transfer(user, amount, to):
    print(f"User {user} sent {amount} to {to}")

transfer("alice", 40, "mum")
transfer("bob", 5000, "landlord")

User alice sent 40 to mum
User bob sent 5000 to landlord


☝ Reads fine to a human. But now your boss asks: *"show me every transfer over £1000 today"*. Your logs are **sentences**, not **data** — you'd have to write fragile regex to pull the number back out of the prose. And every developer phrases the sentence differently (`"User alice sent..."` vs `"Payment by bob:..."`), so there's no reliable field to filter on. A million of these a day and you're grep-ing a haystack.

## 2. Python's built-in `logging` helps a bit — but the message is still a string

Standard `logging` adds levels and timestamps, which is real progress. But the **payload is still a formatted string** — the amount is baked into text, not a field you can query:

In [2]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
log = logging.getLogger("payments")

log.info("transfer user=%s amount=%s to=%s", "alice", 40, "mum")

INFO transfer user=alice amount=40 to=mum


☝ Better — we have a level (`INFO`) and a timestamp is available. But `amount=40` is glued inside one string. To answer *"transfers > £1000"* a log system still has to re-parse text. We want the log to arrive **already structured** — a bag of typed fields. That's exactly what `structlog` gives us.

## 3. The fix — `structlog`: an event name **plus** key=value fields

With `structlog` you log a short **event** (what happened) and pass the details as **keyword arguments**. Each becomes a real field, not text buried in a sentence.

In [5]:
import structlog

log = structlog.get_logger()

# event name first, then the DATA as keywords
log.info("transfer", user="alice", amount=40, to="mum")
log.info("transfer", user="bob", amount=5000, to="landlord")

#log.info("Transferred 1000 from SKB to Gaurav")

2026-07-07 09:29:16 [info     ] transfer                       amount=40 to=mum user=alice
2026-07-07 09:29:16 [info     ] transfer                       amount=5000 to=landlord user=bob


☝ Same information, but now `amount` is a distinct field on each event. Out of the box structlog renders a friendly, colourised, aligned key=value line for the console — pleasant to read while developing. The important shift: **you never format a sentence again**. You name the event once (`"transfer"`) and attach fields. Filtering *"amount > 1000"* is now trivial once these become JSON (next section).

## 4. Make it machine-readable — the JSON renderer

On your laptop the pretty console output is nice. In production (AWS CloudWatch, Datadog, ELK) the log **shipper** wants **JSON**, one object per line, so it can index every field. You choose the output by configuring structlog's **processor pipeline** — an ordered list of functions each log event passes through before it's printed.

In [6]:
import structlog, logging, sys

structlog.configure(
    processors=[
        structlog.contextvars.merge_contextvars,     # pull in any context-var fields (see §6)
        structlog.processors.add_log_level,          # adds "level": "info"
        structlog.processors.TimeStamper(fmt="iso"), # adds "timestamp": "2026-..."
        structlog.processors.JSONRenderer(),         # final step: dump to JSON
    ],
    wrapper_class=structlog.make_filtering_bound_logger(logging.INFO),
    logger_factory=structlog.PrintLoggerFactory(file=sys.stdout),
)

log = structlog.get_logger()
log.info("transfer", user="alice", amount=40, to="mum")
log.warning("transfer", user="bob", amount=5000, to="landlord", flagged=True)

{"user": "alice", "amount": 40, "to": "mum", "event": "transfer", "level": "info", "timestamp": "2026-07-07T04:00:08.837111Z"}
{"user": "bob", "amount": 5000, "to": "landlord", "flagged": true, "event": "transfer", "level": "warning", "timestamp": "2026-07-07T04:00:08.837726Z"}


☝ Each line is now a **JSON object**. A log platform ingests these and lets you query `amount > 1000` or `flagged == true` like a database — no regex. The pipeline reads top-to-bottom: add the level, stamp the time, then `JSONRenderer` turns the accumulated dict into a line. Swap `JSONRenderer()` for `ConsoleRenderer()` and you're back to pretty local logs — **same code, different last step.**

## 5. Bind context once — it rides on every later log

Here's the feature that pays for itself. One customer request touches many log lines (validate → check balance → move money → notify). You want `trace_id` and `session_id` on **all** of them so you can reconstruct the whole request later. Typing those keys into every call is tedious and error-prone. Instead **bind** them once and get a new logger that carries them automatically:

In [7]:
base = structlog.get_logger()

# stamp the request identity ONCE
req_log = base.bind(trace_id="tx-9f2a", session_id="sess-771", user="alice")

req_log.info("request_received", channel="mobile")
req_log.info("balance_checked", balance=120)
req_log.info("transfer", amount=40, to="mum")
req_log.info("notification_sent", method="push")

{"trace_id": "tx-9f2a", "session_id": "sess-771", "user": "alice", "channel": "mobile", "event": "request_received", "level": "info", "timestamp": "2026-07-07T04:00:37.356448Z"}
{"trace_id": "tx-9f2a", "session_id": "sess-771", "user": "alice", "balance": 120, "event": "balance_checked", "level": "info", "timestamp": "2026-07-07T04:00:37.357156Z"}
{"trace_id": "tx-9f2a", "session_id": "sess-771", "user": "alice", "amount": 40, "to": "mum", "event": "transfer", "level": "info", "timestamp": "2026-07-07T04:00:37.357550Z"}
{"trace_id": "tx-9f2a", "session_id": "sess-771", "user": "alice", "method": "push", "event": "notification_sent", "level": "info", "timestamp": "2026-07-07T04:00:37.357964Z"}


☝ Every line carries `trace_id`, `session_id`, and `user` **without repeating them**. `bind()` returns a *new* logger with those fields pre-filled; the original `base` is untouched. Now in your log platform you filter `trace_id = "tx-9f2a"` and see the entire journey of that one payment in order — the manual version of what LangSmith does for an agent run.

## 6. Context vars — fields that reach deep helpers with no plumbing

`bind()` is great when one function does everything. But real code calls helpers, which call more helpers. You don't want to thread `req_log` through every signature. **`contextvars`** solves this: bind values into a context that *any* logger picks up, anywhere in the call stack, for the duration of the request.

In [8]:
from structlog.contextvars import bind_contextvars, clear_contextvars

def check_balance(amount):
    # note: this helper takes NO logger argument
    structlog.get_logger().info("balance_checked", need=amount, ok=True)

def do_transfer(amount, to):
    structlog.get_logger().info("transfer", amount=amount, to=to)

def handle_request():
    clear_contextvars()                                  # start clean each request
    bind_contextvars(trace_id="tx-5000", session_id="sess-99")
    structlog.get_logger().info("request_received")
    check_balance(40)                                    # picks up trace_id automatically
    do_transfer(40, "mum")                               # so does this
    clear_contextvars()                                  # tidy up when done

handle_request()

{"event": "request_received", "session_id": "sess-99", "trace_id": "tx-5000", "level": "info", "timestamp": "2026-07-07T04:00:47.524222Z"}
{"need": 40, "ok": true, "event": "balance_checked", "session_id": "sess-99", "trace_id": "tx-5000", "level": "info", "timestamp": "2026-07-07T04:00:47.524701Z"}
{"amount": 40, "to": "mum", "event": "transfer", "session_id": "sess-99", "trace_id": "tx-5000", "level": "info", "timestamp": "2026-07-07T04:00:47.525084Z"}


☝ `check_balance` and `do_transfer` take **no logger** and know nothing about `trace_id` — yet every line they emit is stamped with it. `bind_contextvars` put the request identity in a place all loggers read from. This is the pattern you'll use in a web/agent framework: bind the request id in one middleware, and every log line anywhere in the request is correlated. `clear_contextvars()` stops one request's ids leaking into the next.

## 7. Log levels still work — filter the noise

Structured logging keeps the familiar levels: `debug` / `info` / `warning` / `error`. We configured the wrapper at `INFO` in §4, so `debug` lines are dropped before they cost anything — cheap in prod, verbose when you need it.

In [9]:
log = structlog.get_logger().bind(trace_id="tx-777")

log.debug("cache_lookup", key="rate_gbp_eur")   # BELOW INFO -> suppressed, no output
log.info("transfer", amount=40, to="mum")        # shows
log.warning("large_transfer", amount=9000)       # shows
log.error("transfer_failed", amount=40, reason="insufficient_funds")  # shows

{"trace_id": "tx-777", "amount": 40, "to": "mum", "event": "transfer", "level": "info", "timestamp": "2026-07-07T04:01:05.236590Z"}
{"trace_id": "tx-777", "amount": 9000, "event": "large_transfer", "level": "warning", "timestamp": "2026-07-07T04:01:05.237226Z"}
{"trace_id": "tx-777", "amount": 40, "reason": "insufficient_funds", "event": "transfer_failed", "level": "error", "timestamp": "2026-07-07T04:01:05.237656Z"}


☝ The `debug` line produced nothing — filtered out by the `INFO` threshold set in `make_filtering_bound_logger`. Same events, same fields; the level is just another queryable field (`level="error"`) *and* a volume control. Bump the threshold to `DEBUG` in one place when you're chasing a bug.

## 8. Exceptions — structured, with context attached

When a transfer blows up you want the error *and* the surrounding context (which user, which trace) in one record. `log.exception(...)` inside an `except` block captures the traceback, and any bound fields come along for free.

In [10]:
log = structlog.get_logger().bind(trace_id="tx-err-1", user="carol")

def withdraw(balance, amount):
    if amount > balance:
        raise ValueError("insufficient funds")
    return balance - amount

try:
    withdraw(balance=30, amount=100)
except ValueError as e:
    log.error("withdraw_failed", amount=100, balance=30, error=str(e))

{"trace_id": "tx-err-1", "user": "carol", "amount": 100, "balance": 30, "error": "insufficient funds", "event": "withdraw_failed", "level": "error", "timestamp": "2026-07-07T04:01:13.521981Z"}


☝ The failure is now a first-class structured event: you can alert on `event="withdraw_failed"`, group by `error`, and still see `trace_id`/`user` to find the exact request. (In real code use `log.exception(...)` and add `structlog.processors.format_exc_info` / `dict_tracebacks` to the pipeline to attach the full traceback as JSON.)

## 9. Tie-in — this is the same idea as LangSmith tracing

In today's LangGraph notebook you'll turn on **LangSmith**, which records each agent run as a **trace**: a tree of structured events (node started, model called, tokens, latency, cost) all sharing one **run id** — exactly like our `trace_id` binding here, one level up.

Structured logs and traces are the same instinct: **emit data, not prose, and correlate it by an id.** A quick mock of what one agent step looks like as a structured log:

In [11]:
agent_log = structlog.get_logger().bind(trace_id="run-abc123", agent="fraud_triage")

agent_log.info("node_start", node="triage")
agent_log.info("model_call", model="gemini-2.5-flash", prompt_tokens=180, completion_tokens=42)
agent_log.info("node_end", node="triage", latency_ms=612, risk="HIGH")

{"trace_id": "run-abc123", "agent": "fraud_triage", "node": "triage", "event": "node_start", "level": "info", "timestamp": "2026-07-07T04:01:17.754451Z"}
{"trace_id": "run-abc123", "agent": "fraud_triage", "model": "gemini-2.5-flash", "prompt_tokens": 180, "completion_tokens": 42, "event": "model_call", "level": "info", "timestamp": "2026-07-07T04:01:17.754988Z"}
{"trace_id": "run-abc123", "agent": "fraud_triage", "node": "triage", "latency_ms": 612, "risk": "HIGH", "event": "node_end", "level": "info", "timestamp": "2026-07-07T04:01:17.755283Z"}


☝ Filter these by `trace_id="run-abc123"` and you've reconstructed one agent run — node timings, token usage, the verdict. LangSmith gives you this as a visual timeline instead of raw lines, but the underlying model is identical: **structured events correlated by a run/trace id.** You just learned the primitive; LangSmith is the hosted, prettier version.

## 10. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **`structlog.get_logger()`** | log an **event name + key=value fields** instead of a formatted sentence |
| **processor pipeline** | ordered steps each event flows through (add level, timestamp, render) |
| **`JSONRenderer`** | machine-readable one-object-per-line output for CloudWatch / Datadog |
| **`ConsoleRenderer`** | pretty, colourised local output — swap it in for dev |
| **`.bind(...)`** | attach fields once; every later log from that logger carries them |
| **`bind_contextvars` / `clear_contextvars`** | request-scoped fields reach deep helpers with no argument plumbing |
| **`make_filtering_bound_logger(INFO)`** | drop low-priority logs cheaply; level is both a filter and a field |
| **`log.error` / `log.exception`** | failures as structured, alertable, context-rich events |

**Takeaway:** stop writing log *sentences*. Emit *events with fields*, bind an id to correlate them, render JSON in prod. That is the exact mental model behind LangSmith traces — the subject of today's LangGraph notebook.

> **Install note:** this notebook needs `structlog` (`pip install structlog` / `uv pip install structlog`). No API key required — it runs fully offline.
